<a href="https://colab.research.google.com/github/vivianlinnn/DS41_IDXExchange/blob/main/src/06_FeatureEngineering_DecisionTree_Team.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# START : Anjali Manju Gowda

# Objective
The goal of this notebook is to build and evaluate a **Decision Tree Regressor** for predicting residential property closing prices. We aim to explore how feature engineering and careful preprocessing can improve model performance.

## Why Decision Trees?
- **Interpretability:** Trees generate simple "if-then" rules that are easy to understand and explain to stakeholders, such as real estate agents or investors.  
- **Non-linearity:** They can capture complex, non-linear relationships between features and property prices without explicit transformations.  
- **Automatic Feature Selection:** Decision trees inherently focus on the most important features at each split, helping identify influential predictors.

## Key Features Considered

| Feature Type | Features | Description |
|--------------|----------|-------------|
| Original numeric features | LivingArea, BedroomsTotal, BathroomsTotalInteger, LotSizeSquareFeet, Property_Age | Core property attributes |
| Engineered features | Bed_Bath_Ratio, Sqft_Per_Bedroom, Lot_Utilization | Derived from numeric features to capture bedroom-bathroom balance, space per bedroom, and lot efficiency |
| Location-based feature | ZipMedianPrice | Median property price per ZIP code, capturing neighborhood-level effects |
| Time-related feature | Months_From_Dec_2025 | Captures timing of sale relative to December 2025 |
| Target-derived feature (excluded) | PPSF (Price per Square Foot) | Calculated from `ClosePrice`; excluded to prevent overfitting |

## Target Variable
- `ClosePrice` (log-transformed as `LogClosePrice` to stabilize variance and handle skewness)

## Evaluation Metrics
- **R² (Coefficient of Determination):** Measures variance explained by the model  
- **MAPE (Mean Absolute Percentage Error):** Average percentage error in predictions  
- **MdAPE (Median Absolute Percentage Error):** Median percentage error, robust to outliers

## Notebook Workflow
1. Data loading and cleaning  
2. Feature engineering (including engineered numeric features, location encoding, and time-based features)  
3. Train-test split and model training  
4. Prediction and evaluation of performance  
5. Export and interpretation of decision tree rules  
6. Comparison between previous and retrained models
#================================

## Import Libraries and Load Data

This section imports the core Python libraries required for data manipulation, modeling, and evaluation.

- **pandas** and **numpy** are used for data cleaning, numerical operations, and feature engineering.
- **DecisionTreeRegressor** from `sklearn.tree` is used to train the regression model that predicts property prices.
- **export_text** allows us to extract interpretable rules from the trained decision tree.
- **r2_score** and **mean_absolute_percentage_error** are used to evaluate model performance.

The training and testing datasets (`train_cleaned_fe.csv` and `test_cleaned_fe.csv`) are loaded using:
- `engine='python'` to improve compatibility with irregular CSV formatting.
- `on_bad_lines='skip'` to safely skip malformed rows and prevent loading errors.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.metrics import r2_score, mean_absolute_percentage_error

#Load CSV

train = pd.read_csv('train_cleaned_fe.csv', engine='python', on_bad_lines='skip')
test  = pd.read_csv('test_cleaned_fe.csv', engine='python', on_bad_lines='skip')

## Convert Columns to Numeric Format

Real-world datasets often contain mixed data types or corrupted values that prevent proper numerical analysis.

This step ensures that all relevant numeric columns are properly converted to numeric format using:

`pd.to_numeric(..., errors='coerce')`

If a value cannot be converted to a number, it is replaced with **NaN**.

Key variables converted include:
- **LivingArea** – total interior square footage
- **BedroomsTotal** – number of bedrooms
- **BathroomsTotalInteger** – number of bathrooms
- **LotSizeSquareFeet** – lot size
- **Property_Age** – age of the property
- **Months_From_Dec_2025** – temporal feature
- **Bed_Bath_Ratio** – bedroom-to-bathroom relationship

Rows where the **target variable (`ClosePrice`) is missing** are removed to ensure the model is trained only on valid observations.

In [ ]:
#Convert numeric columns safely
numeric_cols = [
    'LivingArea','BedroomsTotal','LotSizeSquareFeet',
    'BathroomsTotalInteger','Bed_Bath_Ratio',
    'Property_Age','Months_From_Dec_2025'
]

for col in numeric_cols:
    train[col] = pd.to_numeric(train[col], errors='coerce')
    test[col]  = pd.to_numeric(test[col], errors='coerce')

train['ClosePrice'] = pd.to_numeric(train['ClosePrice'], errors='coerce')
test['ClosePrice']  = pd.to_numeric(test['ClosePrice'], errors='coerce')

train = train.dropna(subset=['ClosePrice'])
test  = test.dropna(subset=['ClosePrice'])

## Log Transformation of Target Variable

Housing prices typically exhibit **right-skewed distributions**, where a small number of expensive properties dominate the price range.

To stabilize variance and improve model performance, we transform the target variable:

\[
LogClosePrice = log(ClosePrice)
\]

Benefits of log transformation include:

- Reducing the influence of extreme price values
- Making the distribution more symmetric
- Improving model stability and predictive accuracy

Predictions will later be converted back to the original dollar scale using the exponential function.

In [ ]:
#Create log price
train['LogClosePrice'] = np.log(train['ClosePrice'])
test['LogClosePrice']  = np.log(test['ClosePrice'])

## Feature Engineering

Feature engineering creates additional variables that capture meaningful relationships between existing features.

Before generating new features, zero values in **BedroomsTotal** and **LotSizeSquareFeet** are replaced with NaN to prevent division errors.

Two engineered features are then created:

**1. Sqft_Per_Bedroom**

\[
Sqft\_Per\_Bedroom = \frac{LivingArea}{BedroomsTotal}
\]

This captures how much living space is allocated per bedroom, which may reflect home layout quality.

**2. Lot_Utilization**

\[
Lot\_Utilization = \frac{LivingArea}{LotSizeSquareFeet}
\]

This measures how efficiently the available land is used.

These features help the model understand **space efficiency and property density**, which are important drivers of real estate value.

# PPSF (Price per Square Foot)
- is intentionally NOT included in tree_features
- Reason: PPSF is calculated using the target variable (ClosePrice), which would allow the model to trivially "cheat" and overfit.
- Including it would not reflect true predictive performance.

In [ ]:
#Feature engineering
for df in [train, test]:

    df['BedroomsTotal'] = df['BedroomsTotal'].replace(0, np.nan)
    df['LotSizeSquareFeet'] = df['LotSizeSquareFeet'].replace(0, np.nan)

    df['Sqft_Per_Bedroom'] = df['LivingArea'] / df['BedroomsTotal']
    df['Lot_Utilization']  = df['LivingArea'] / df['LotSizeSquareFeet']

## ZIP Code Median Price Encoding

Location plays a critical role in real estate pricing. Instead of using ZIP codes directly as categorical variables, we encode them using a **median price signal**.

Steps performed:

1. Calculate the **median log closing price** for each ZIP code in the training dataset.
2. Map this value to each property based on its ZIP code.
3. Assign the value to a new feature called **ZipMedianPrice**.

This approach captures **neighborhood-level pricing patterns** while avoiding the high dimensionality of one-hot encoding.

If a ZIP code appears in the test set but not in the training set, it is assigned the **global median log price** to prevent missing values.

In [ ]:
#ZIP median encoding
zip_median_price = train.groupby('PostalCode')['LogClosePrice'].median()

train['ZipMedianPrice'] = train['PostalCode'].map(zip_median_price)
test['ZipMedianPrice']  = test['PostalCode'].map(zip_median_price)

global_median = train['LogClosePrice'].median()
test['ZipMedianPrice'] = test['ZipMedianPrice'].fillna(global_median)

## Feature Selection

This section defines the set of features used to train the decision tree model.

The selected features combine:

### Core Property Attributes
- **LivingArea**
- **BedroomsTotal**
- **BathroomsTotalInteger**

### Location Information
- **ZipMedianPrice**

### Engineered Features
- **Bed_Bath_Ratio**
- **Sqft_Per_Bedroom**
- **Lot_Utilization**

### Time and Age Factors
- **Property_Age**
- **Months_From_Dec_2025**

All features are converted to numeric format to ensure compatibility with the machine learning model.

===========================================
# ZIP Median Price Feature
# -------------------------------------------
-ZipMedianPrice is a location-based feature that encodes the median property closing price within each ZIP code.
-It captures neighborhood-level price signals such as:
 - local market value
 - school district quality
 - demand and amenities
-This numeric feature allows the model to use location
-information without high-dimensional one-hot encoding of ZIP codes.
# ===========================================


In [ ]:
#Features
tree_features = [
    'ZipMedianPrice',
    'LivingArea',
    'BathroomsTotalInteger',
    'BedroomsTotal',
    'Bed_Bath_Ratio',
    'Property_Age',
    'Months_From_Dec_2025',
    'Sqft_Per_Bedroom',
    'Lot_Utilization'
]

for col in tree_features:
    train[col] = pd.to_numeric(train[col], errors='coerce')
    test[col]  = pd.to_numeric(test[col], errors='coerce')

## Handling Missing and Infinite Values

Machine learning algorithms require clean numerical inputs.

This step ensures that the dataset contains only valid values by:

1. Replacing infinite values (`inf` and `-inf`) with **NaN**
2. Dropping rows that contain missing values in:
   - Any selected feature
   - The target variable (`LogClosePrice`)

This prevents training errors and ensures the model learns from reliable data.

In [ ]:
#Remove NaN / inf
for df in [train, test]:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

train = train.dropna(subset=tree_features + ['LogClosePrice'])
test  = test.dropna(subset=tree_features + ['LogClosePrice'])

## Prepare Feature Matrices and Target Variables

The dataset is separated into:

- **X (features)** → input variables used for prediction
- **y (target)** → the variable the model is trying to predict

Variables created:

- **X_train** and **X_test**: feature matrices
- **y_train** and **y_test**: target values (`LogClosePrice`)

Features are converted to **float32** to improve memory efficiency and computational performance.

In [ ]:
#Prepare X and y
X_train = train[tree_features].astype(np.float32)
X_test  = test[tree_features].astype(np.float32)

y_train = train['LogClosePrice']
y_test  = test['LogClosePrice']

## Train Decision Tree Regressor

A Decision Tree Regressor is trained to predict the **log-transformed closing price**.

Regularization parameters are used to control model complexity and reduce overfitting:

- **max_depth = 6**  
  Limits how deep the tree can grow.

- **min_samples_leaf = 100**  
  Ensures each leaf contains a minimum number of samples, preventing overly specific splits.

- **criterion = 'squared_error'**  
  Uses mean squared error to determine the best split.

Decision trees are particularly useful because they can capture **nonlinear relationships** between housing features and prices.

In [ ]:
#Train faster Decision Tree
tree_model = DecisionTreeRegressor(
    max_depth=6,
    min_samples_leaf=100,
    criterion='squared_error',
    random_state=42
)

tree_model.fit(X_train, y_train)

DecisionTreeRegressor(max_depth=6, min_samples_leaf=100, random_state=42)

## Generate Predictions

After training the model, predictions are generated for both datasets:

- **train_pred** → predictions on training data
- **test_pred** → predictions on testing data

These predictions represent the **log-transformed property prices**.

They will later be converted back to dollar values to calculate meaningful error metrics.

In [ ]:
#Predictions
train_pred = tree_model.predict(X_train)
test_pred  = tree_model.predict(X_test)

## Model Performance Evaluation

Several metrics are used to evaluate model performance.

### 1. R² (Coefficient of Determination)

Measures the proportion of variance in housing prices explained by the model.

\[
R^2 = 1 - \frac{SS_{res}}{SS_{tot}}
\]

Higher values indicate better model performance.

### 2. Mean Absolute Percentage Error (MAPE)

Measures the average percentage difference between predicted and actual prices.

### 3. Median Absolute Percentage Error (MdAPE)

Similar to MAPE but uses the **median**, making it more robust to outliers.

Because the model was trained on **log prices**, predictions are converted back to the **original dollar scale using `exp()`** before calculating percentage errors.

The resulting metrics provide insight into both **model accuracy and robustness**.

In [ ]:
#Evaluation
train_r2 = r2_score(y_train, train_pred)
test_r2  = r2_score(y_test, test_pred)

y_train_d = np.exp(y_train)
y_test_d  = np.exp(y_test)

train_pred_d = np.exp(train_pred)
test_pred_d  = np.exp(test_pred)

train_mape = mean_absolute_percentage_error(y_train_d, train_pred_d) * 100
test_mape  = mean_absolute_percentage_error(y_test_d, test_pred_d) * 100

train_mdape = np.median(np.abs((y_train_d-train_pred_d)/y_train_d))*100
test_mdape  = np.median(np.abs((y_test_d-test_pred_d)/y_test_d))*100

print(f"Train R²: {train_r2:.4f}, Test R²: {test_r2:.4f}")
print(f"Train MAPE: {train_mape:.2f}%, Test MAPE: {test_mape:.2f}%")
print(f"Train MdAPE: {train_mdape:.2f}%, Test MdAPE: {test_mdape:.2f}%")

Train R²: 0.8727, Test R²: 0.8511
Train MAPE: 16.53%, Test MAPE: 17.76%
Train MdAPE: 12.25%, Test MdAPE: 12.77%


## Extract Interpretable Decision Tree Rules

One advantage of decision trees is their **interpretability**.

Using `export_text`, we extract the tree structure as human-readable rules such as:


In [ ]:
#Tree Rules
rules = export_text(tree_model, feature_names=tree_features)
print(rules)

|--- ZipMedianPrice <= 13.78
|   |--- ZipMedianPrice <= 13.32
|   |   |--- LivingArea <= 1804.50
|   |   |   |--- ZipMedianPrice <= 13.06
|   |   |   |   |--- ZipMedianPrice <= 12.83
|   |   |   |   |   |--- LivingArea <= 1389.50
|   |   |   |   |   |   |--- value: [12.55]
|   |   |   |   |   |--- LivingArea >  1389.50
|   |   |   |   |   |   |--- value: [12.74]
|   |   |   |   |--- ZipMedianPrice >  12.83
|   |   |   |   |   |--- LivingArea <= 1318.50
|   |   |   |   |   |   |--- value: [12.77]
|   |   |   |   |   |--- LivingArea >  1318.50
|   |   |   |   |   |   |--- value: [12.93]
|   |   |   |--- ZipMedianPrice >  13.06
|   |   |   |   |--- BedroomsTotal <= 2.50
|   |   |   |   |   |--- ZipMedianPrice <= 13.22
|   |   |   |   |   |   |--- value: [12.93]
|   |   |   |   |   |--- ZipMedianPrice >  13.22
|   |   |   |   |   |   |--- value: [13.04]
|   |   |   |   |--- BedroomsTotal >  2.50
|   |   |   |   |   |--- ZipMedianPrice <= 13.24
|   |   |   |   |   |   |--- value: [13.09]
| 

## Results: Decision Tree Model Performance

The Decision Tree Regressor achieved strong predictive performance on both the training and testing datasets. The evaluation metrics are summarized below:

| Metric | Training | Testing |
|------|------|------|
| R² | 0.8708 | 0.8499 |
| MAPE | 16.65% | 17.91% |
| MdAPE | 12.37% | 13.13% |

### Model Interpretation

**1. High Variance Explained**

The model explains approximately **87% of the variance in training data** and **85% in the test data**.  
The small difference between train and test R² indicates that the model generalizes well and does not significantly overfit.

**2. Prediction Error**

- The **Mean Absolute Percentage Error (MAPE)** shows that predictions are on average within **~17–18%** of the true property price.
- The **Median Absolute Percentage Error (MdAPE)** is lower (~12–13%), meaning that the typical prediction error is relatively small and that some higher errors are likely due to extreme property values.

**3. Model Stability**

The close alignment between training and testing metrics suggests that the model is **stable and robust**.  
The use of regularization parameters (`max_depth=6`, `min_samples_leaf=100`) helped prevent overfitting while maintaining good predictive accuracy.

### Overall Performance

The results demonstrate that the Decision Tree model, combined with engineered features and ZIP-level median price encoding, can effectively capture relationships between property characteristics and closing prices.

This model provides a good balance between **predictive accuracy and interpretability**, making it suitable for real estate price estimation and market analysis.

# END : Anjali Manju Gowda